# 02 · Expense category classifier

Trains and evaluates three candidate models on expense descriptions:

1. Logistic Regression
2. Multinomial Naive Bayes
3. Random Forest

The shared preprocessing is a TF-IDF vectoriser over the description plus an
amount bucket (see `backend/ml/feature_utils.py`). The best model by weighted F1
is persisted with Joblib to `backend/ml/expense_classifier.pkl`.

Run this notebook from the `notebooks/` directory, or adjust `PROJECT_ROOT`
manually.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "ml"))
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import json

import pandas as pd


In [ ]:
import train_classifier as tc

df = tc.load_and_clean()
print(f"Usable rows: {len(df):,}")
display(df[["description", "amount", "category", "text"]].head())


In [ ]:
df["category"].value_counts()


## Train all candidates

`tc.main()` performs the stratified 80/20 split, fits the three pipelines,
selects the best weighted-F1 model, writes the metrics JSON, saves the
confusion-matrix plot and persists the Joblib bundle used by the Flask API.


In [ ]:
tc.main()


## Model comparison

In [ ]:
metrics = json.loads((PROJECT_ROOT / "backend" / "ml" / "classifier_metrics.json").read_text(encoding="utf-8"))
comparison = pd.DataFrame(metrics["comparison"]).T
comparison


In [ ]:
report = pd.DataFrame(metrics["classification_report"]).T
report


## Confusion matrix

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(PROJECT_ROOT / "backend" / "ml" / "confusion_matrix.png")))


## Interpretability: strongest TF-IDF coefficients

In [ ]:
top = metrics.get("top_features", {})
if top and all(isinstance(value, dict) for value in top.values()):
    for label, features in top.items():
        print(f"\n{label}")
        display(pd.Series(features).sort_values(ascending=False))
elif top:
    display(pd.Series(top).sort_values(ascending=False))
else:
    print("The selected model does not expose linear coefficients.")


## Live prediction check

The same bundle is loaded by `backend/services/classifier_service.py`. This
cell mirrors the API's text preparation so the notebook and production path
stay aligned.


In [ ]:
import joblib
from feature_utils import build_classifier_text

bundle = joblib.load(PROJECT_ROOT / "backend" / "ml" / "expense_classifier.pkl")
pipeline = bundle["pipeline"]

samples = [
    ("Swiggy dinner order", 480),
    ("Uber ride to office", 260),
    ("Electricity bill payment", 1850),
    ("PVR movie tickets", 900),
    ("Apollo Pharmacy medicines", 640),
]

for description, amount in samples:
    text = build_classifier_text(description, amount, "Expense")
    prediction = pipeline.predict([text])[0]
    confidence = float(pipeline.predict_proba([text]).max())
    print(f"{description:32s} -> {prediction:14s} ({confidence:.1%})")


## Notes for the report

- Accuracy can be close to 1.0 because the synthetic descriptions are strongly
  associated with categories. This is a property of the reproducible dataset,
  not evidence of real-world perfection.
- The API stores every prediction in the `predictions` table. When a user
  changes the suggested category, `was_corrected = 1` is recorded so future
  retraining can use real corrections.
- The classifier only runs for **expense** transactions. Income categories are
  selected directly by the user.
